# Processamento em Lote (Batch) - Sinais Periféricos (GSR e HR) | Tarefa: Stroop
Extração em lote das métricas fisiológicas fásicas (GSR e HR) para a amostra completa (n=54) durante a tarefa de controlo inibitório.
Inclui a sincronização temporal via `.snirf` para mapear os inícios dos blocos experimentais baseados em Schroeter et al. (2002):
* `1` = Bloco Congruente
* `2` = Bloco Incongruente

A janela temporal foi ajustada (-2s a 15s) para captar a resposta fisiológica ao bloco cognitivo. São calculadas a amplitude máxima de suor (stress simpático) e a bradicardia máxima (foco/desaceleração) para ambas as condições. A rotina exclui automaticamente sujeitos com falhas de hardware no sinal de Frequência Cardíaca (HR) (e.g., `sub-042`, `sub-047`).

In [1]:
import os
import zipfile
import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt
import warnings
import mne

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

# ==========================================
# 1. Configuração do Lote e Parâmetros Base
# ==========================================
BASE_DIR = r"C:\Users\guilh\Desktop\OneDrive - Universidade de Coimbra\PhD_Project_ARDITI_Guilherme Saldanha\Study2_Data Collection Files_PhD_Guilherme Saldanha_UC-UMa-ITI"
NUM_SUJEITOS = 54 
TAREFA = "tk-stroop" 

JANELA_INICIO_S = -2.0
JANELA_FIM_S = 15.0

TRIGGER_CONGRUENTE = '1'
TRIGGER_INCONGRUENTE = '2'

# Blacklists (Com base nas tuas anotações do laboratório)
SUJEITOS_SEM_HR = ['sub-042', 'sub-047'] 
SUJEITOS_SEM_GSR = []         

dados_mestre_fisio_stroop = []

print(f"A iniciar processamento periférico (STROOP) em lote para {NUM_SUJEITOS} sujeitos...\n")

# ==========================================
# 2. Ciclo de Processamento
# ==========================================
for i in range(1, NUM_SUJEITOS + 1):
    sujeito_id = f"sub-{i:03d}"
    pasta_nirs = os.path.join(BASE_DIR, sujeito_id, "nirs")
    
    try:
        if not os.path.exists(pasta_nirs):
            raise FileNotFoundError("Pasta 'nirs' não existe.")
            
        pastas_dentro_nirs = os.listdir(pasta_nirs)
        nome_pasta_run = next((p for p in pastas_dentro_nirs if TAREFA in p.lower() and os.path.isdir(os.path.join(pasta_nirs, p))), None)
        
        if not nome_pasta_run:
            raise FileNotFoundError(f"Pasta da tarefa '{TAREFA}' não encontrada.")
            
        pasta_alvo = os.path.join(pasta_nirs, nome_pasta_run)
        
        ficheiros_pasta = os.listdir(pasta_alvo)
        snirf_file = next((f for f in ficheiros_pasta if f.endswith('.snirf')), None)
        zip_file = next((f for f in ficheiros_pasta if f.endswith('.zip')), None)
        
        if not snirf_file or not zip_file:
            raise FileNotFoundError("Ficheiros .snirf ou .zip em falta.")
            
        caminho_snirf = os.path.join(pasta_alvo, snirf_file)
        caminho_zip = os.path.join(pasta_alvo, zip_file)
        pasta_extraida_wings = os.path.join(pasta_alvo, "dados_wings_extraidos")
        
        if not os.path.exists(pasta_extraida_wings):
            with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
                zip_ref.extractall(pasta_extraida_wings)
                
        ficheiros_extraidos = os.listdir(pasta_extraida_wings)
        wings_file = next((f for f in ficheiros_extraidos if f.endswith('.wings')), None)
        
        if not wings_file:
             raise FileNotFoundError("Ficheiro .wings não extraído.")
             
        caminho_wings = os.path.join(pasta_extraida_wings, wings_file)

        # -- Sincronização (.snirf) --
        raw_temp = mne.io.read_raw_snirf(caminho_snirf, preload=False, verbose=False)
        
        tempos_congruente = [onset for onset, desc in zip(raw_temp.annotations.onset, raw_temp.annotations.description) if desc == TRIGGER_CONGRUENTE]
        tempos_incongruente = [onset for onset, desc in zip(raw_temp.annotations.onset, raw_temp.annotations.description) if desc == TRIGGER_INCONGRUENTE]
                
        if len(tempos_congruente) == 0 or len(tempos_incongruente) == 0:
            print(f"  [{sujeito_id}] AVISO: Marcadores de blocos em falta (Con: {len(tempos_congruente)}, Inc: {len(tempos_incongruente)}). A saltar...")
            continue
            
        df_wings = pd.read_csv(caminho_wings, sep=';', skiprows=1)
        df_wings.columns = df_wings.columns.str.strip()
        
        tempos_wings = df_wings['SecondsSinceRunStart'].values
        fs_wings = len(tempos_wings) / tempos_wings[-1]
        
        indices_congruente = [np.argmin(np.abs(tempos_wings - t)) for t in tempos_congruente]
        indices_incongruente = [np.argmin(np.abs(tempos_wings - t)) for t in tempos_incongruente]
        
        amostras_antes = int(abs(JANELA_INICIO_S) * fs_wings)
        amostras_depois = int(JANELA_FIM_S * fs_wings)

        # Função auxiliar de extração
        def extrair_metrica(sinal_bruto, indices, tipo='gsr'):
            if tipo == 'gsr':
                b, a = butter(4, 5.0 / (0.5 * fs_wings), btype='low', analog=False)
                sinal_filtrado = filtfilt(b, a, sinal_bruto)
            else:
                sinal_filtrado = sinal_bruto
                
            epocas = []
            for idx in indices:
                if idx - amostras_antes >= 0 and idx + amostras_depois < len(sinal_filtrado):
                    sinal = sinal_filtrado[idx - amostras_antes : idx + amostras_depois]
                    epocas.append(sinal - np.mean(sinal[0:amostras_antes]))
                    
            if not epocas: return np.nan, np.nan
            media_total = np.mean(np.array(epocas), axis=0)
            
            if tipo == 'gsr':
                return np.mean(media_total), np.max(media_total)
            else:
                return np.mean(media_total), np.min(media_total)

        # -- Processamento GSR --
        if sujeito_id not in SUJEITOS_SEM_GSR:
            gsr_bruto = df_wings['GSR'].values
            gsr_media_con, gsr_amp_con = extrair_metrica(gsr_bruto, indices_congruente, 'gsr')
            gsr_media_inc, gsr_amp_inc = extrair_metrica(gsr_bruto, indices_incongruente, 'gsr')
        else:
            gsr_media_con, gsr_amp_con, gsr_media_inc, gsr_amp_inc = np.nan, np.nan, np.nan, np.nan

        # -- Processamento HR --
        if sujeito_id not in SUJEITOS_SEM_HR:
            hr_bruto = df_wings['HR'].values
            hr_media_con, hr_bradicardia_con = extrair_metrica(hr_bruto, indices_congruente, 'hr')
            hr_media_inc, hr_bradicardia_inc = extrair_metrica(hr_bruto, indices_incongruente, 'hr')
        else:
            hr_media_con, hr_bradicardia_con, hr_media_inc, hr_bradicardia_inc = np.nan, np.nan, np.nan, np.nan

        # -- Adicionar à BD --
        dados_mestre_fisio_stroop.append({
            'ID': sujeito_id,
            'Tarefa': 'Stroop',
            'GSR_Media_Con': gsr_media_con,
            'GSR_Amplitude_Con': gsr_amp_con,
            'GSR_Media_Inc': gsr_media_inc,
            'GSR_Amplitude_Inc': gsr_amp_inc,
            'HR_Media_Con': hr_media_con,
            'HR_Bradicardia_Con': hr_bradicardia_con,
            'HR_Media_Inc': hr_media_inc,
            'HR_Bradicardia_Inc': hr_bradicardia_inc
        })
        print(f"  [{sujeito_id}] OK (Con: {len(tempos_congruente)}, Inc: {len(tempos_incongruente)} blocos).")
        
    except Exception as e:
        print(f"  [{sujeito_id}] ERRO: {e}")

# ==========================================
# 3. Exportação da Base de Dados Consolidada
# ==========================================
if dados_mestre_fisio_stroop:
    df_mestre = pd.DataFrame(dados_mestre_fisio_stroop)
    pasta_exportacao = os.path.join(BASE_DIR, "derivatives", "mne-nirs")
    os.makedirs(pasta_exportacao, exist_ok=True)
    caminho_csv_final = os.path.join(pasta_exportacao, "dataset_mestre_fisio_STROOP.csv")
    df_mestre.to_csv(caminho_csv_final, index=False)
    print(f"\nProcessamento concluído! Dados extraídos de {len(dados_mestre_fisio_stroop)} sujeitos.")

A iniciar processamento periférico (STROOP) em lote para 54 sujeitos...

  [sub-001] OK (Con: 5, Inc: 5 blocos).
  [sub-002] OK (Con: 5, Inc: 5 blocos).
  [sub-003] OK (Con: 5, Inc: 5 blocos).
  [sub-004] OK (Con: 5, Inc: 5 blocos).
  [sub-005] OK (Con: 5, Inc: 5 blocos).
  [sub-006] OK (Con: 5, Inc: 5 blocos).
  [sub-007] OK (Con: 5, Inc: 5 blocos).
  [sub-008] OK (Con: 5, Inc: 5 blocos).
  [sub-009] OK (Con: 5, Inc: 5 blocos).
  [sub-010] OK (Con: 5, Inc: 5 blocos).
  [sub-011] OK (Con: 5, Inc: 5 blocos).
  [sub-012] OK (Con: 5, Inc: 5 blocos).
  [sub-013] OK (Con: 5, Inc: 5 blocos).
  [sub-014] OK (Con: 5, Inc: 5 blocos).
  [sub-015] OK (Con: 5, Inc: 5 blocos).
  [sub-016] OK (Con: 5, Inc: 5 blocos).
  [sub-017] OK (Con: 5, Inc: 5 blocos).
  [sub-018] OK (Con: 5, Inc: 5 blocos).
  [sub-019] OK (Con: 5, Inc: 5 blocos).
  [sub-020] OK (Con: 5, Inc: 5 blocos).
  [sub-021] OK (Con: 5, Inc: 5 blocos).
  [sub-022] OK (Con: 5, Inc: 5 blocos).
  [sub-023] OK (Con: 5, Inc: 5 blocos).
  [sub-